In [2]:
!pip show tensorflow

Name: tensorflow
Version: 2.20.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: C:\Users\yohan\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, tensorboard, termcolor, typing_extensions, wrapt
Required-by: 


In [3]:
! pip install tensorflow


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import re
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
import joblib
import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [5]:
# Load the dataset menggunakan relative path
df = pd.read_json('./data/clean_recipes_5000.json')

# Tampilkan 5 baris pertama untuk melihat struktur data awal
print("Struktur data awal:")
display(df.head())


Struktur data awal:


,Title,Ingredients,Steps,Loves,URL,Category,Title Cleaned,Total Ingredients,Ingredients Cleaned,Total Steps,judul_bersih,bahan_bersih,Quality Score
0,Chilli Tuna Puff Kilat Super Yummy,1/2 pack Kulit puff instant saya merk.Edo--Bah...,1) Langkah \n1.Tumis chili tuna chunk dengan m...,516,https://cookpad.com/id/resep/4463240-chilli-tu...,ikan,chilli tuna puff kilat super yummy,9,"pack kulit puff instant merkedo , isian , kale...",3,chilli tuna puff kilat super yummy,1 2 pack kulit puff instant saya merk edo baha...,0.4689
1,Perkedel Tahu Simple,3 buah tahu petak--1 batang daun seledri--2 si...,1) Giling halus bawang merah+bawang putih+ mer...,481,https://cookpad.com/id/resep/4337985-perkedel-...,tahu,perkedel tahu simple,8,"tahu petak , batang daun seledri , bawang puti...",5,perkedel tahu simple,3 buah tahu petak 1 batang daun seledri 2 siun...,0.4827
2,Orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera--3 buah cab...,1) Goreng tempe yg sdh di potong2 dlm minyak p...,452,https://cookpad.com/id/resep/3989566-orek-temp...,tempe,orek tempe basah bumbu ulek,12,"papan tempe potong , cabe ijo , kecap manis , ...",3,orek tempe basah bumbu ulek,1 papan tempe potong sesuai selera 3 buah cabe...,0.4223
3,Sop Iga Sapi Enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan--Secukupn...","1) Didihkan secukupnya air, lalu masukan poton...",375,https://cookpad.com/id/resep/3310336-sop-iga-s...,sapi,sop iga sapi enaaak bangeet,22,"iga sapi , tiriskan , air didihkan utk rebusan...",4,sop iga sapi enaaak bangeet,"1 kg iga sapi, cuci bersih, tiriskan secukupny...",0.3550
4,Nasi Goreng Kambing...ala kebon sirih,3 piring nasi putih dingin--350 gr daging kamb...,"1) Tumis kapulaga, bumbu halus, bubuk kari sam...",355,https://cookpad.com/id/resep/1041289-nasi-gore...,kambing,nasi goreng kambing ala kebon sirih,14,"piring nasi putih dingin , daging kambing , po...",5,nasi goreng kambing ala kebon sirih,3 piring nasi putih dingin 350 gr daging kambi...,0.4114


In [6]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Title                4999 non-null   str    
 1   Ingredients          4999 non-null   str    
 2   Steps                4999 non-null   str    
 3   Loves                4999 non-null   int64  
 4   URL                  4999 non-null   str    
 5   Category             4999 non-null   str    
 6   Title Cleaned        4999 non-null   str    
 7   Total Ingredients    4999 non-null   int64  
 8   Ingredients Cleaned  4999 non-null   str    
 9   Total Steps          4999 non-null   int64  
 10  judul_bersih         4999 non-null   str    
 11  bahan_bersih         4999 non-null   str    
 12  Quality Score        4999 non-null   float64
dtypes: float64(1), int64(3), str(9)
memory usage: 507.8 KB


Title                  0
Ingredients            0
Steps                  0
Loves                  0
URL                    0
Category               0
Title Cleaned          0
Total Ingredients      0
Ingredients Cleaned    0
Total Steps            0
judul_bersih           0
bahan_bersih           0
Quality Score          0
dtype: int64

In [7]:
df.describe()

,Loves,Total Ingredients,Total Steps,Quality Score
count,4999.000000,4999.000000,4999.000000,4999.000000
mean,21.816563,12.507902,5.447890,0.299901
std,27.765506,4.615280,2.219461,0.068369
min,6.000000,3.000000,2.000000,0.084200
25%,9.000000,9.000000,4.000000,0.255600
50%,12.000000,12.000000,5.000000,0.293400
75%,25.000000,16.000000,7.000000,0.337600
max,516.000000,25.000000,23.000000,0.739700


In [8]:
import json
from collections import Counter

# Load data resep
with open('./data/clean_recipes_5000.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Hitung distribusi kategori
categories = [item.get('Category', 'Unknown') for item in data]
category_counts = Counter(categories)

print("Distribusi Kategori:")
for cat, count in category_counts.items():
    print(f"{cat}: {count}")

# Cek apakah seimbang
total = len(data)
print(f"\nTotal resep: {total}")
print("Kategori mayoritas:", category_counts.most_common(1)[0][0], "dengan", category_counts.most_common(1)[0][1], "resep")
print("Kategori minoritas:", category_counts.most_common()[-1][0], "dengan", category_counts.most_common()[-1][1], "resep")

# Hitung rasio
majority = category_counts.most_common(1)[0][1]
minority = category_counts.most_common()[-1][1]
ratio = majority / minority
print(f"Rasio mayoritas/minoritas: {ratio:.2f}")

if ratio > 2:
    print("Dataset tidak seimbang, mungkin ada bias terhadap kategori mayoritas.")
else:
    print("Dataset cukup seimbang.")

Distribusi Kategori:
ikan: 625
tahu: 625
tempe: 625
sapi: 624
kambing: 625
telur: 625
ayam: 625
udang: 625

Total resep: 4999
Kategori mayoritas: ikan dengan 625 resep
Kategori minoritas: sapi dengan 624 resep
Rasio mayoritas/minoritas: 1.00
Dataset cukup seimbang.


In [9]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

# Create the 'ingredients_clean' column, copied from cell g8lhkxZS_fxc
df['ingredients_clean'] = df['Ingredients Cleaned'].apply(clean_text)


In [10]:

# Fitur numerik
num_features = ['Total Ingredients', 'Total Steps', 'Loves']
X_num = df[num_features].fillna(0).values

# Fitur teks (TF-IDF)
vectorizer = TfidfVectorizer(max_features=150, stop_words='english', min_df=2)
X_tfidf = vectorizer.fit_transform(df['ingredients_clean']).toarray()

# Gabungkan semua fitur (tanpa Category sebagai input, karena akan menjadi target)
X = np.hstack([X_num, X_tfidf])
print(f"Total fitur (X): {X.shape}")

# Target (Category, untuk klasifikasi)
# Gunakan LabelEncoder untuk mengubah kategori string menjadi angka integer
le_category = LabelEncoder()
y = le_category.fit_transform(df['Category'])
print(f"Target (y) shape: {y.shape}")
print(f"Mapping Kategori: {list(le_category.classes_)}")

Total fitur (X): (4999, 153)
Target (y) shape: (4999,)
Mapping Kategori: ['ayam', 'ikan', 'kambing', 'sapi', 'tahu', 'telur', 'tempe', 'udang']


In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

Train: (3499, 153), Validation: (750, 153), Test: (750, 153)


In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaling selesai. Contoh nilai pertama train:\n", X_train_scaled[0])

Scaling selesai. Contoh nilai pertama train:
 [-0.98499336  0.23717342 -0.40261898 -0.80136881 -0.17431954 -0.24363402
 -0.24148837 -0.5280141  -0.34612953 -0.28302795 -0.1666918  -0.18396289
 -0.32458301 -0.43572579 -0.13621519 -0.17553289 -0.20835447 -0.49216394
  1.95355454 -0.18851336 -0.14262133 -0.33704545 -0.79404049 -0.20210687
 -0.27624687 -0.18758173 -0.25435106 -0.4978695  -0.87137701 -0.12393693
 -0.20864256 -0.13816895 -0.13859369 -0.15442149  1.49480624 -0.31370577
 -0.14986999 -0.5609984   4.86570682 -0.60734144 -0.24308478 -0.29918087
 -0.16876391 -0.13935841 -0.14900577 -0.33138613 -0.14536585  3.5315649
 -0.22933486 -0.24734014 -0.55980331 -0.20153705 -0.19918885 -0.43271793
 -0.36075483 -0.17882104 -0.20340963 -0.19902722 -0.22332652 -0.59562778
 -0.16163891 -0.18354563 -0.19002648 -0.47966519 -0.14530854 -0.16578208
 -0.25747719 -0.13824932 -0.34802939 -0.45641561 -0.1749443  -0.2047773
 -0.22868192 -0.16615174 -0.20342887 -0.17305612  3.58370339 -0.22034101
 -0.495

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.utils import register_keras_serializable

# 1. Definisikan Custom Layer dari model_mlp.py
@register_keras_serializable()
class IngredientsImportanceLayer(Layer):
    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs):
        return inputs * self.factor

# 2. Definisikan Custom Loss Function
@register_keras_serializable()
def custom_recipe_loss(y_true, y_pred):
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(loss)

# 3. Build Model (Sesuai dengan fungsi build_mlp_model)
input_dim = X_train_scaled.shape[1]  # Dinamis menyesuaikan 503 fitur
num_classes = len(le_category.classes_) # Dinamis menyesuaikan 8 kategori

input_layer = Input(shape=(input_dim,), name='input')
x = IngredientsImportanceLayer()(input_layer)

x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

output_layer = Dense(num_classes, activation='softmax', name='output')(x)

# Buat models
model = Model(inputs=input_layer, outputs=output_layer, name='SayurKita_Classifier')
model.compile(optimizer='adam', loss=custom_recipe_loss, metrics=['accuracy'])
model.summary()


Model: "SayurKita_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 153)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ingredients_importance_layer    │ (None, 153)            │             0 │
│ (IngredientsImportanceLayer)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,488 (111.28 KB)

 Trainable params: 28,488 (111.28 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
import tensorflow as tf

class StopAtAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, target=0.90):
        super().__init__()
        self.target = target
    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = logs.get('val_accuracy')
        if val_accuracy and val_accuracy >= self.target: # Periksa apakah lebih besar atau sama dengan target
            print(f"\n Target Akurasi {self.target} tercapai di epoch {epoch+1}. Stop training.")
            self.model.stop_training = True

In [15]:
import tensorflow as tf

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[
        StopAtAccuracy(target=0.90), 
        tf.keras.callbacks.EarlyStopping(
            patience=8, 
            restore_best_weights=True, 
            monitor='val_loss', 
            mode='min'
        )
    ],
    verbose=1
)

Epoch 1/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.2366 - loss: 2.1985 - val_accuracy: 0.5613 - val_loss: 1.5326
Epoch 2/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4701 - loss: 1.4799 - val_accuracy: 0.7133 - val_loss: 1.0766
Epoch 3/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6167 - loss: 1.1179 - val_accuracy: 0.8040 - val_loss: 0.7684
Epoch 4/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7148 - loss: 0.8499 - val_accuracy: 0.8320 - val_loss: 0.6003
Epoch 5/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.7822 - loss: 0.6649 - val_accuracy: 0.8467 - val_loss: 0.5207
Epoch 6/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8165 - loss: 0.5652 - val_accuracy: 0.8640 - val_loss: 0.4607
Epoch 7/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8408 - loss: 0.4772 - val_accuracy: 0.8787 - val_loss: 0.4312
Epoch 8/100
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8608 - loss: 0.4092 - val_accuracy: 0

In [16]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
if test_accuracy >= 0.85: # Sesuaikan target untuk akurasi
    print("Target Akurasi ≥ 0.85 tercapai pada test set!")
else:
    print(f"Akurasi masih {test_accuracy:.4f}, perlu perbaikan.")

Test Accuracy: 0.8827
Target Akurasi ≥ 0.85 tercapai pada test set!


In [17]:
import datetime

# Buat nama file dengan timestamp untuk menghindari overwrite
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
model_save_path = f"./SayurKita_Classifier_.keras" # Tambahkan ekstensi .keras

# Simpan model
tf.keras.models.save_model(model, model_save_path)
print(f"Model berhasil disimpan di: {model_save_path}")

Model berhasil disimpan di: ./SayurKita_Classifier_.keras


In [18]:
# Load the saved model
loaded_model = tf.keras.models.load_model(model_save_path)

# Create a dummy input for demonstration (e.g., using the first test sample)
dummy_input = X_test_scaled[0].reshape(1, -1) # Reshape for a single sample

# Make a prediction
predictions = loaded_model.predict(dummy_input)
predicted_class_index = np.argmax(predictions, axis=1)[0]
predicted_category = le_category.inverse_transform([predicted_class_index])[0]

print(f"Dummy input shape: {dummy_input.shape}")
print(f"Prediction probabilities: {predictions[0]}")
print(f"Predicted class index: {predicted_class_index}")
print(f"Predicted category: {predicted_category}")

# You can compare this with the actual category for the first test sample
actual_class_index = y_test[0]
actual_category = le_category.inverse_transform([actual_class_index])[0]
print(f"Actual category for the first test sample: {actual_category}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step
Dummy input shape: (1, 153)
Prediction probabilities: [2.4973208e-01 7.4598038e-01 7.0655486e-05 1.0500278e-03 9.1588154e-04
 6.1218085e-04 1.4925306e-03 1.4633246e-04]
Predicted class index: 1
Predicted category: ikan
Actual category for the first test sample: ikan
